# Surface-density fly-through

Camera path over one snapshot, making a perspective surface-density map per frame. Example paths: inclined circular orbit, orbit with radial drift.


In [ ]:
from pathlib import Path

import matplotlib.colors as colors
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.insert(0, str(Path("..").resolve()))

from simviz.field_plots import project_surface_density_camera
from simviz.projections import rotate_about_axis, rotate_to_bar_frame
from simviz.utils import read_snapshot_hdf5

EXAMPLE_OUT = Path("../example_output")
EXAMPLE_OUT.mkdir(exist_ok=True)

print("imports done")

In [ ]:
# Gas snapshot: positions + masses only (MHD not required).
snap_path = Path("/Users/hph/Research/Archive/Old_Code/arepo_CMZ/TS_2020/bar_sink_sne_250.hdf5")

data, header = read_snapshot_hdf5(snap_path)
print(f"Loaded: {snap_path}")
print(f"N gas cells: {len(data['Coordinates']):,}")
print("PartType0 fields:", sorted(data.keys()))

In [ ]:
# GC-center; bar frame 
t = float(header["Time"])
box = float(header["BoxSize"])

x, y, z = np.asarray(data["Coordinates"], dtype=np.float64).T
masses = np.asarray(data["Masses"], dtype=np.float64)

x -= box / 2.0
y -= box / 2.0
z -= box / 2.0

zeros = np.zeros_like(x)
x, y, _, _ = rotate_to_bar_frame(x, y, zeros, zeros, t)

print(f"Prepared {x.size:,} particles in bar frame")

In [ ]:
def inclined_orbit_path(
    n_frames,
    radius=8.0,
    tilt_axis=(1.0, 1.0, 0.3),
    tilt_deg=35.0,
    z_offset=0.0,
    center=(0.0, 0.0, 0.0),
):
    #Camera path: circular orbit rotated about a non-trivial axis.
    theta = np.linspace(0.0, 2.0 * np.pi, n_frames, endpoint=False)
    base = np.column_stack([
        radius * np.cos(theta),
        radius * np.sin(theta),
        np.full(n_frames, z_offset),
    ])
    tilted = rotate_about_axis(base, axis=tilt_axis, angle=np.radians(tilt_deg))
    return tilted + np.asarray(center, dtype=np.float64)


def orbit_with_radial_drift_path(
    n_frames,
    r_start=12.0,
    r_end=6.0,
    n_turns=1.5,
    tilt_axis=(1.0, 1.0, 0.3),
    tilt_deg=35.0,
    center=(0.0, 0.0, 0.0),
):
    #Camera path: orbit plus radial in/out drift.
    theta = np.linspace(0.0, 2.0 * np.pi * n_turns, n_frames, endpoint=False)
    radius = np.linspace(r_start, r_end, n_frames)
    base = np.column_stack([
        radius * np.cos(theta),
        radius * np.sin(theta),
        np.zeros(n_frames),
    ])
    tilted = rotate_about_axis(base, axis=tilt_axis, angle=np.radians(tilt_deg))
    return tilted + np.asarray(center, dtype=np.float64)

In [ ]:
def render_surface_density_frame(
    x,
    y,
    z,
    masses,
    camera_position,
    target=(0.0, 0.0, 0.0),
    up_hint=(0.0, 0.0, 1.0),
    fov_x_deg=55.0,
    nx=700,
    ny=700,
    z_near=0.2,
    z_far=40.0,
    vmin=1e-3,
    vmax=2e1,
    cmap="Greys",
    ax=None,
):
    sigma, extent = project_surface_density_camera(
        x,
        y,
        z,
        masses,
        camera_position=camera_position,
        target=target,
        up_hint=up_hint,
        fov_x_deg=fov_x_deg,
        nx=nx,
        ny=ny,
        z_near=z_near,
        z_far=z_far,
    )

    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(6, 6), dpi=150)
    else:
        fig = ax.figure

    im = ax.imshow(
        sigma,
        origin="lower",
        extent=extent,
        norm=colors.LogNorm(vmin=vmin, vmax=vmax),
        cmap=cmap,
        interpolation="nearest",
    )
    ax.set_xlabel("camera x")
    ax.set_ylabel("camera y")
    return fig, ax, im

In [ ]:
# Single-frame preview
path = inclined_orbit_path(n_frames=120, radius=10.0, tilt_deg=35.0)

i = 0
cam = path[i]
fig, ax, im = render_surface_density_frame(
    x,
    y,
    z,
    masses,
    camera_position=cam,
    target=(0.0, 0.0, 0.0),
    up_hint=(0.0, 0.0, 1.0),
    fov_x_deg=55.0,
    z_near=0.2,
    z_far=30.0,
    vmin=5e-4,
    vmax=2e1,
)
fig.colorbar(im, ax=ax, pad=0.02, label=r"Surface density [code mass / pixel]")
ax.set_title(f"Preview frame {i}: cam=({cam[0]:.2f}, {cam[1]:.2f}, {cam[2]:.2f})")
fig.savefig(EXAMPLE_OUT / "surface_density_preview.png", dpi=180, bbox_inches="tight")
plt.show()

## Focal-plane column density

`project_surface_density_camera` returns **mass per image bin** in normalized camera coordinates. To report surface density on a **reference plane at distance D** along the camera forward direction: divide each bin by the physical pixel area on that plane,

\[
A_{\rm pix} = \frac{2 D \tan(\mathrm{FOV}_x/2)}{n_x}\,\frac{2 D \tan(\mathrm{FOV}_y/2)}{n_y},
\]

using the same horizontal/vertical FOV convention as the projector (vertical FOV follows aspect ratio `ny/nx` when not set explicitly). Resulting \(\Sigma\) has units **code mass / (code length)\(^2\)** for positions and masses in code units; multiply by your UnitLength/UnitMass factors if you need physical \(M_\odot\,{\rm pc}^{-2}\).

**Adaptive GC distance:** if the camera looks at the GC (`target` at origin), choosing \(D = |\mathrm{camera} - \mathrm{target}|\) puts the focal plane through that point each frame (distance scales with orbit radius).

In [ ]:
# Reference plane at distance D (code length units): Σ = (mass per LOS bin) / pixel area on that plane.
# D: "gc" -> |camera - target| (focal plane through look-at, e.g. GC); "fixed" -> FOCAL_DISTANCE_FIXED.
FOCAL_PLANE_MODE = "gc"  # "gc" | "fixed"
FOCAL_DISTANCE_FIXED = 8.0  # used when FOCAL_PLANE_MODE == "fixed"
TARGET_GC = (0.0, 0.0, 0.0)
FOV_X_DEG = 55.0
NX = 700
NY = 700


def focal_plane_pixel_area(focal_distance, fov_x_deg, nx, ny, fov_y_deg=None):
    """Physical pixel area on the focal plane (matches ``project_surface_density_camera`` binning)."""
    tx = np.tan(np.radians(fov_x_deg) / 2.0)
    if fov_y_deg is None:
        fov_y_deg = fov_x_deg * (ny / float(nx))
    ty = np.tan(np.radians(fov_y_deg) / 2.0)
    width = 2.0 * focal_distance * tx
    height = 2.0 * focal_distance * ty
    return (width / nx) * (height / ny)


def focal_distance_for_plane(camera_position, target, mode, fixed_distance):
    """Choose D along the view axis for pixel area (see markdown: adaptive GC)."""
    if mode == "gc":
        cam = np.asarray(camera_position, dtype=np.float64)
        tgt = np.asarray(target, dtype=np.float64)
        return float(np.linalg.norm(cam - tgt))
    return float(fixed_distance)


def mass_map_to_column_density(mass_per_pixel, focal_distance, fov_x_deg, nx, ny, fov_y_deg=None):
    """Convert mass bins from ``project_surface_density_camera`` to Σ on the focal plane."""
    area = focal_plane_pixel_area(focal_distance, fov_x_deg, nx, ny, fov_y_deg=fov_y_deg)
    return mass_per_pixel / area


def render_column_density_frame(
    x,
    y,
    z,
    masses,
    camera_position,
    focal_distance,
    target=(0.0, 0.0, 0.0),
    up_hint=(0.0, 0.0, 1.0),
    fov_x_deg=55.0,
    nx=700,
    ny=700,
    z_near=0.2,
    z_far=40.0,
    vmin=1e-5,
    vmax=1e3,
    cmap="Greys",
    ax=None,
):
    sigma_mass, extent = project_surface_density_camera(
        x,
        y,
        z,
        masses,
        camera_position=camera_position,
        target=target,
        up_hint=up_hint,
        fov_x_deg=fov_x_deg,
        nx=nx,
        ny=ny,
        z_near=z_near,
        z_far=z_far,
    )
    sigma = mass_map_to_column_density(sigma_mass, focal_distance, fov_x_deg, nx, ny)

    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(6, 6), dpi=150)
    else:
        fig = ax.figure

    im = ax.imshow(
        sigma,
        origin="lower",
        extent=extent,
        norm=colors.LogNorm(vmin=vmin, vmax=vmax),
        cmap=cmap,
        interpolation="nearest",
    )
    ax.set_xlabel("camera x")
    ax.set_ylabel("camera y")
    return fig, ax, im


# Same geometry as the single-frame preview above
path_col = inclined_orbit_path(n_frames=120, radius=10.0, tilt_deg=35.0)
i = 0
cam = path_col[i]
D_frame = focal_distance_for_plane(cam, TARGET_GC, FOCAL_PLANE_MODE, FOCAL_DISTANCE_FIXED)
fig, ax, im = render_column_density_frame(
    x,
    y,
    z,
    masses,
    camera_position=cam,
    focal_distance=D_frame,
    target=TARGET_GC,
    up_hint=(0.0, 0.0, 1.0),
    fov_x_deg=FOV_X_DEG,
    nx=NX,
    ny=NY,
    z_near=0.2,
    z_far=30.0,
    vmin=1e-4,
    vmax=1e3,
)
apix = focal_plane_pixel_area(D_frame, FOV_X_DEG, NX, NY)
fig.colorbar(
    im,
    ax=ax,
    pad=0.02,
    label=r"$\Sigma$ [code mass / (code length)$^2$]",
)
ax.set_title(
    f"Focal plane D={D_frame:.3f} ({FOCAL_PLANE_MODE}): frame {i}, "
    f"cam=({cam[0]:.2f}, {cam[1]:.2f}, {cam[2]:.2f})\n"
    f"pixel area = {apix:.4g} (code length)$^2$"
)
fig.savefig(EXAMPLE_OUT / "surface_density_preview_focal_plane.png", dpi=180, bbox_inches="tight")
plt.show()

### Optional: full PNG sequence with focal-plane \(\Sigma\)

Uses the same radial-drift camera path as the cell below. Outputs `flythrough_frames_focal/frame_%04d.png` so you can encode a separate MP4 without overwriting mass-per-pixel frames.

In [ ]:
# Same path as the mass-per-pixel sequence; tune vmin/vmax for Σ if needed.
out_dir_focal = Path("flythrough_frames_focal")
out_dir_focal.mkdir(exist_ok=True)

path_focal = orbit_with_radial_drift_path(n_frames=180, r_start=12.0, r_end=6.0, n_turns=1.5)

for i, cam in enumerate(path_focal):
    D_frame = focal_distance_for_plane(cam, TARGET_GC, FOCAL_PLANE_MODE, FOCAL_DISTANCE_FIXED)
    fig, ax, im = render_column_density_frame(
        x,
        y,
        z,
        masses,
        camera_position=cam,
        focal_distance=D_frame,
        target=TARGET_GC,
        up_hint=(0.0, 0.0, 1.0),
        fov_x_deg=FOV_X_DEG,
        nx=NX,
        ny=NY,
        z_near=0.2,
        z_far=30.0,
        vmin=1e-4,
        vmax=1e3,
    )
    ax.set_title(f"frame {i:04d}  (D={D_frame:.3f}, mode={FOCAL_PLANE_MODE})")
    fig.savefig(out_dir_focal / f"frame_{i:04d}.png", bbox_inches="tight")
    plt.close(fig)

print(f"Wrote {len(path_focal)} focal-plane frames to: {out_dir_focal.resolve()}")

In [ ]:
# PNG sequence (encode video separately)
out_dir = Path("flythrough_frames")
out_dir.mkdir(exist_ok=True)

# Alternate camera paths:
# path = inclined_orbit_path(n_frames=180, radius=10.0, tilt_deg=35.0)
path = orbit_with_radial_drift_path(n_frames=180, r_start=12.0, r_end=6.0, n_turns=1.5)

for i, cam in enumerate(path):
    fig, ax, im = render_surface_density_frame(
        x,
        y,
        z,
        masses,
        camera_position=cam,
        target=(0.0, 0.0, 0.0),
        up_hint=(0.0, 0.0, 1.0),
        fov_x_deg=55.0,
        z_near=0.2,
        z_far=30.0,
        vmin=5e-4,
        vmax=2e1,
    )
    ax.set_title(f"frame {i:04d}")
    fig.savefig(out_dir / f"frame_{i:04d}.png", bbox_inches="tight")
    plt.close(fig)

print(f"Wrote {len(path)} frames to: {out_dir.resolve()}")

In [ ]:
# MP4: run from this directory (same folder as flythrough_frames/ or flythrough_frames_focal/). libx264+yuv420p needs even width and height; bbox_inches='tight' can produce odd sizes, so crop to even pixels first:
# ffmpeg -y -framerate 30 -i flythrough_frames/frame_%04d.png -vf "scale=trunc(iw/2)*2:trunc(ih/2)*2" -c:v libx264 -pix_fmt yuv420p surface_density_flythrough.mp4
# ffmpeg -y -framerate 30 -i flythrough_frames_focal/frame_%04d.png -vf "scale=trunc(iw/2)*2:trunc(ih/2)*2" -c:v libx264 -pix_fmt yuv420p surface_density_flythrough_focal.mp4
